# 1. Data Processing

## Importing data

In [1]:
import os

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import lightgbm as lgb
import shap
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = "dataset/"

files = {}
for f in os.listdir(DATA_PATH):
    if f.endswith(".csv"):
        name = f.replace(".csv", "")
        files[name] = pd.read_csv(DATA_PATH + f)
        print(f"{f:35s} → {files[name].shape}")

customers.csv                       → (121930, 7)
geography.csv                       → (39948, 4)
inventory.csv                       → (60247, 17)
orders.csv                          → (646945, 8)
order_items.csv                     → (714669, 7)
payments.csv                        → (646945, 4)
products.csv                        → (2412, 8)
promotions.csv                      → (50, 10)
returns.csv                         → (39939, 7)
reviews.csv                         → (113551, 7)
sales.csv                           → (3833, 3)
sample_submission.csv               → (548, 3)
shipments.csv                       → (566067, 4)
web_traffic.csv                     → (3652, 7)


## Preprocessing

In [3]:
customers = files['customers']
geography = files['geography']
inventory = files['inventory']
orders = files['orders']
order_items = files['order_items']
payments = files['payments']
products = files['products']
promotions = files['promotions']
returns = files['returns']
reviews = files['reviews']
sales = files['sales']
shipments = files['shipments']
web_traffic = files['web_traffic']

In [4]:
# Convert date columns to datetime format

inventory['snapshot_date'] = pd.to_datetime(inventory['snapshot_date'])

orders['order_date'] = pd.to_datetime(orders['order_date'])

promotions['start_date'] = pd.to_datetime(promotions['start_date'])
promotions['end_date'] = pd.to_datetime(promotions['end_date'])

returns['return_date'] = pd.to_datetime(returns['return_date'])

reviews['review_date'] = pd.to_datetime(reviews['review_date'])

sales['Date'] = pd.to_datetime(sales['Date'])

shipments['ship_date'] = pd.to_datetime(shipments['ship_date'])
shipments['delivery_date'] = pd.to_datetime(shipments['delivery_date'])

web_traffic['date'] = pd.to_datetime(web_traffic['date'])

In [5]:
TEST_START = pd.to_datetime("2023-01-01")
TEST_END = pd.to_datetime("2024-07-01")

df = pd.DataFrame({'date': sales['Date']})

In [6]:
sales['gross_margin'] = (sales['Revenue'] - sales['COGS']) / sales['Revenue']

df = df.merge(sales, left_on='date', right_on='Date', how='left')
df.drop(columns='Date', inplace=True)

In [7]:
df

,date,Revenue,COGS,gross_margin
0,2012-07-04,5123547.94,3982991.19,0.222611
1,2012-07-05,2751773.45,2150580.23,0.218475
2,2012-07-06,3054029.42,2517632.84,0.175636
3,2012-07-07,2667930.94,2108246.62,0.209782
4,2012-07-08,2360851.90,1808622.79,0.233911
...,...,...,...,...
3828,2022-12-27,2100553.66,2184872.24,-0.040141
3829,2022-12-28,3448729.20,3513621.00,-0.018816
3830,2022-12-29,3083944.33,3170787.10,-0.028160
3831,2022-12-30,2884668.76,3022292.15,-0.047709
